# Chapter 6: Multiple Regression Analysis — Further Issues

Topics: scaling, beta coefficients, logarithms, polynomials, prediction intervals, interaction terms.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('wage1')

## 6.1.3 Logarithms

In [ ]:
# Log-level model
m_log = smf.ols('np.log(wage) ~ educ + exper + tenure', data=df).fit()
print(m_log.summary())
print(f"\nReturns to education: {m_log.params['educ']*100:.2f}% per year")

## 6.1.4 Quadratics and Polynomials

In [ ]:
m_quad = smf.ols('np.log(wage) ~ educ + exper + I(exper**2) + tenure', data=df).fit()
print(m_quad.summary())

# Turning point: d(log wage)/d(exper) = b_exper + 2*b_exper2*exper = 0
b1 = m_quad.params['exper']
b2 = m_quad.params['I(exper ** 2)']
turning_point = -b1 / (2 * b2)
print(f"\nTurning point of experience: {turning_point:.1f} years")

# Plot marginal effect
exper_range = np.linspace(0, 40, 100)
marginal = b1 + 2 * b2 * exper_range
plt.figure(figsize=(8, 4))
plt.plot(exper_range, marginal, color='steelblue', lw=2)
plt.axhline(0, color='red', linestyle='--')
plt.axvline(turning_point, color='green', linestyle='--', label=f'Turning point: {turning_point:.1f} yrs')
plt.xlabel('Experience (years)'); plt.ylabel('Marginal effect on log(wage)')
plt.title('Marginal Effect of Experience (Quadratic Model)')
plt.legend()
plt.savefig(f"{output_dir}/fig1_quadratic_exper.png", dpi=150)
plt.show()

## 6.1.6 Interaction Terms

In [ ]:
m_interact = smf.ols('np.log(wage) ~ educ * female + exper + tenure', data=df).fit()
print(m_interact.summary())

b_educ        = m_interact.params['educ']
b_female_educ = m_interact.params['educ:female']
print(f"\nReturn to education for men:   {b_educ*100:.2f}%")
print(f"Return to education for women: {(b_educ + b_female_educ)*100:.2f}%")

## 6.2 Prediction and Confidence Intervals

In [ ]:
new_obs = pd.DataFrame({'educ': [16], 'exper': [5], 'tenure': [2]})
pred = m_log.get_prediction(new_obs)
pred_df = pred.summary_frame(alpha=0.05)
print("Prediction for: educ=16, exper=5, tenure=2")
print(pred_df[['mean','mean_ci_lower','mean_ci_upper','obs_ci_lower','obs_ci_upper']])
print(f"\nPredicted log(wage): {pred_df['mean'].values[0]:.4f}")
print(f"Predicted wage:      ${np.exp(pred_df['mean'].values[0]):.2f}/hr")